# 01 — Kalenderdata Integratie
## Masterproef: Spatiotemporal Prediction and Optimization of Car Parking in Mechelen

---

### Doel van dit notebook

Dit notebook consolideert alle kalenderdata (feestdagen en schoolvakanties) tot één 
uniforme master-kalender die als externe feature-bron dient voor het voorspellingsmodel.

---

### Academische onderbouwing

Temporele kalendereffecten behoren tot de meest consistente externe factoren in 
parkeerbezettigingsonderzoek. Focke et al. (2021) tonen aan dat feestdagen en 
evenementen kritische exogene variabelen zijn voor SARIMAX-modellen bij langetermijn 
parkeerprognoses. Zhang et al. (2020) integreren expliciete event-mechanismen in hun 
PewLSTM-architectuur en bereiken daarmee een nauwkeurigheid van 93,84% — tegenover 
lagere scores zonder kalendercomponent.

Specifiek voor schoolvakanties geldt dat zij structurele verschuivingen in 
herhalingspatronen veroorzaken: tijdens vakantieperiodes daalt het aandeel 
woon-werkverkeer significant, terwijl recreatief parkeren toeneemt (Niu et al., 2023; 
Zhao et al., 2024). Dit maakt het onderscheid tussen `is_school_vacation` en 
`is_national_holiday` relevant als aparte binaire features.

In de context van Mechelen — een compacte binnenstad met vijf centrumparkings, vijf 
vestenparkings en één randparking — zijn feestdagen en vakanties extra kritisch omdat 
winkel-, horeca- en toerismebezoekers de dominante gebruikerssegmenten vormen voor 
kortetermijnparkeerders.

---

### Input
| Bestand | Locatie | Beschrijving |
|---|---|---|
| `20XX_national_fd.csv` | `data_raw/` | Officiële Belgische feestdagen per jaar (2020–2025) |
| `20XX_other_fd.csv` | `data_raw/` | Overige feestdagen / herdenkingsdagen per jaar (2020–2025) |
| `schoolvakanties_belgie_2020_2025.csv` | `data_raw/` | Schoolvakanties Vlaanderen in periodeformaat |

### Output
| Bestand | Locatie | Beschrijving |
|---|---|---|
| `calendar_holidays_daily.parquet` | `data_intermediate/` | Feestdagen geëxplodeerd naar dag-per-rij |
| `calendar_schoolvac_daily.parquet` | `data_intermediate/` | Schoolvakanties geëxplodeerd naar dag-per-rij |
| `calendar_master.parquet` | `data_intermediate/` | Gecombineerde master-kalender (1 rij per dag) |


In [1]:
import pandas as pd
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif PROJECT_ROOT.parent.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parents[1]

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.project_config import DEFAULT_PROJECT_END, DEFAULT_PROJECT_START, get_project_paths
from src.io.calendar_readers import list_holiday_files, read_holiday_csv, read_school_vacations_csv
from src.phase1.calendar import build_calendar_master, load_holiday_files, prepare_school_vacations

paths = get_project_paths(PROJECT_ROOT)
DATA_RAW = paths.data_raw
DATA_INT = paths.data_intermediate

print(f"ROOT        : {paths.root}")
print(f"DATA_RAW    : {DATA_RAW}")
print(f"DATA_INT    : {DATA_INT}")



ROOT        : /Users/emilevandevoorde/Documents/mp_mechelen_parking_v2
DATA_RAW    : /Users/emilevandevoorde/Documents/mp_mechelen_parking_v2/data_raw
DATA_INT    : /Users/emilevandevoorde/Documents/mp_mechelen_parking_v2/data_intermediate


## Stap 1 — Feestdagen laden & samenvoegen

De feestdagen zijn beschikbaar als 12 losse CSV-bestanden (6 jaar × 2 types: 
`national` en `other`). We laden deze parallel in via `glob`, voegen een 
`holiday_type`-kolom toe als metadata, en concateneren tot één dataframe.

**Aandachtspunt:** De datumkolom is geformatteerd als `"01 Jan 2020"` 
(Engelstalig maandnaam-formaat) en vereist expliciete parsering via `dayfirst=False`.


In [2]:
# -- Alle feestdagen CSV's inladen ---------------------------------------------
national_files, other_files = list_holiday_files(DATA_RAW)

print(f"National bestanden gevonden : {len(national_files)}")
print(f"Other bestanden gevonden    : {len(other_files)}")

# -- Peek: inspecteer structuur van eerste bestand ------------------------------
df_peek = read_holiday_csv(national_files[0])
print("--- Voorbeeld structuur (2020national_fd.csv) ---")
print(df_peek.head(3))
print(f"Kolommen: {df_peek.columns.tolist()}")



National bestanden gevonden : 6
Other bestanden gevonden    : 6
--- Voorbeeld structuur (2020national_fd.csv) ---
             Feestdag        Datum
0  Nieuwjaarsdag 2020  01 Jan 2020
1     1e Paasdag 2020  12 Apr 2020
2     2e Paasdag 2020  13 Apr 2020
Kolommen: ['Feestdag', 'Datum']


### Kolomnormalisatie

De ruwe kolommen zijn `['Feestdag', 'Datum']`. Voor consistentie hernoemen we 
deze naar Engelstalige, snake_case-namen: `holiday_name` en `date`. 
Dit is een best practice in data engineering: vermijd niet-ASCII kolomnamen 
in analytische pipelines (Pandas documentatie).


In [3]:
df_national = load_holiday_files(national_files, "national")
df_other = load_holiday_files(other_files, "other")

df_holidays = pd.concat([df_national, df_other], ignore_index=True)
df_holidays = df_holidays.sort_values("date").reset_index(drop=True)

print(f"Totaal feestdagen (rauw)   : {len(df_holidays)}")
print(f"Datumbereik                : {df_holidays['date'].min().date()} -> {df_holidays['date'].max().date()}")
print(f"Types                      : {df_holidays['holiday_type'].value_counts().to_dict()}")
print("\n", df_holidays.head(6))


Totaal feestdagen (rauw)   : 150
Datumbereik                : 2020-01-01 -> 2025-12-31
Types                      : {'other': 78, 'national': 72}

          holiday_name       date holiday_type  year
0  Nieuwjaarsdag 2020 2020-01-01     national  2020
1  Drie Koningen 2020 2020-01-06        other  2020
2  Valentijnsdag 2020 2020-02-14        other  2020
3  Goede vrijdag 2020 2020-04-10        other  2020
4     1e Paasdag 2020 2020-04-12     national  2020
5     2e Paasdag 2020 2020-04-13     national  2020


### Deduplicatie

Sommige feestdagen overlappen tussen `national` en `other` 
(bijv. Goede Vrijdag staat soms in beide lijsten). We dedupliceren op 
`(date, holiday_type)` combinaties. Als een dag zowel nationaal als 'other' is, 
bewaren we beide rijen — ze dragen immers verschillende semantische informatie.


In [4]:
before = len(df_holidays)
df_holidays = df_holidays.drop_duplicates(subset=["date", "holiday_type"])
after = len(df_holidays)

print(f"Rijen voor deduplicatie  : {before}")
print(f"Rijen na deduplicatie    : {after}")
print(f"Verwijderd               : {before - after} duplicaten")

# ── Validatie: verwachte aantallen per jaar ──────────────────────────────────
print("\nFeestdagen per jaar & type:")
print(df_holidays.groupby(["year","holiday_type"]).size().unstack(fill_value=0))


Rijen voor deduplicatie  : 150
Rijen na deduplicatie    : 150
Verwijderd               : 0 duplicaten

Feestdagen per jaar & type:
holiday_type  national  other
year                         
2020                12     13
2021                12     13
2022                12     13
2023                12     13
2024                12     13
2025                12     13


In [5]:
# opslaan
out_path = DATA_INT / "calendar_holidays_daily.parquet"
df_holidays.to_parquet(out_path, index=False)
print(f"✓ Opgeslagen: {out_path}  ({len(df_holidays)} rijen)")


✓ Opgeslagen: /Users/emilevandevoorde/Documents/mp_mechelen_parking_v2/data_intermediate/calendar_holidays_daily.parquet  (150 rijen)


## Stap 2 — Schoolvakanties laden & exploderen naar dagformaat

De schoolvakanties zijn beschikbaar als periodes (start–eind). Voor een directe 
merge op dagelijkse tijdstempels moeten we elke vakantieperiode "exploderen" 
naar individuele dagen.

**Regio-filtering:** België kent drie onderwijsgemeenschappen met licht afwijkende 
vakantiekalenders (Vlaams, Frans, Duits). Mechelen valt onder de Vlaamse 
Gemeenschap. We filteren op `Vlaanderen` (of equivalent) indien de regio-kolom 
beschikbaar is.

**Randgeval:** Vakanties die jaargrens overschrijden (bijv. Kerstvakantie 
2021–2022) worden correct verwerkt door `pd.date_range(start, end)`.


In [6]:
# laden en inspecteren
VAC_FILE = DATA_RAW / "schoolvakanties_belgie_2020_2025.csv"
df_vac_raw = read_school_vacations_csv(VAC_FILE)

print(f"Shape  : {df_vac_raw.shape}")
print(f"Kolommen: {df_vac_raw.columns.tolist()}")
print("\n", df_vac_raw.head(8))


Shape  : (42, 5)
Kolommen: ['Vakantie', 'Type', 'Jaar', 'Startdatum', 'Einddatum']

                   Vakantie                Type  Jaar  Startdatum   Einddatum
0       Kerstvakantie 2019       Kerstvakantie  2019  2019-12-23  2020-01-05
1      Krokusvakantie 2020      Krokusvakantie  2020  2020-02-24  2020-03-01
2        Paasvakantie 2020        Paasvakantie  2020  2020-04-06  2020-04-19
3  Hemelvaartvakantie 2020  Hemelvaartvakantie  2020  2020-05-21  2020-05-22
4       Zomervakantie 2020       Zomervakantie  2020  2020-07-01  2020-08-31
5      Herfstvakantie 2020      Herfstvakantie  2020  2020-11-02  2020-11-15
6       Kerstvakantie 2020       Kerstvakantie  2020  2020-12-21  2021-01-03
7       Kerstvakantie 2020       Kerstvakantie  2020  2020-12-21  2021-01-03


In [7]:
df_vac_daily = prepare_school_vacations(df_vac_raw)

print(f"Dagelijkse vakantierijen : {len(df_vac_daily)}")
print(f"Datumbereik              : {df_vac_daily['date'].min().date()} -> {df_vac_daily['date'].max().date()}")
print("\n", df_vac_daily.head(8))


Dagelijkse vakantierijen : 663
Datumbereik              : 2019-12-23 -> 2026-01-04

         date       vacation_name  vacation_type
0 2019-12-23  Kerstvakantie 2019  Kerstvakantie
1 2019-12-24  Kerstvakantie 2019  Kerstvakantie
2 2019-12-25  Kerstvakantie 2019  Kerstvakantie
3 2019-12-26  Kerstvakantie 2019  Kerstvakantie
4 2019-12-27  Kerstvakantie 2019  Kerstvakantie
5 2019-12-28  Kerstvakantie 2019  Kerstvakantie
6 2019-12-29  Kerstvakantie 2019  Kerstvakantie
7 2019-12-30  Kerstvakantie 2019  Kerstvakantie


In [8]:
# opslaan
out_path = DATA_INT / "calendar_schoolvac_daily.parquet"
df_vac_daily.to_parquet(out_path, index=False)
print(f"✓ Opgeslagen: {out_path}  ({len(df_vac_daily)} rijen)")


✓ Opgeslagen: /Users/emilevandevoorde/Documents/mp_mechelen_parking_v2/data_intermediate/calendar_schoolvac_daily.parquet  (663 rijen)


## Stap 3 — Master-kalender bouwen

We bouwen een volledige dagkalender van **1 januari 2019 t/m 31 december 2026** 
(dekking van de parkeerdata). Vervolgens joinen we de feestdagen en schoolvakanties 
op deze basis-datumreeks.

Het resultaat is een calendar lookup table: voor elke datum in de parkeerdata 
kan men via een LEFT JOIN op `date_only` direct de kalenderkenmerken ophalen.

### Feature-ontwerp

Op basis van Niu et al. (2023), Zhao et al. (2024) en Fokker et al. (2021) 
definiëren we de volgende binaire en categorische features:

| Feature | Type | Beschrijving |
|---|---|---|
| `is_national_holiday` | binary (0/1) | Officiële Belgische feestdag |
| `is_other_holiday` | binary (0/1) | Overige herdenkingsdagen |
| `is_any_holiday` | binary (0/1) | Unie van bovenstaande twee |
| `is_school_vacation` | binary (0/1) | Schoolvakantiedag (Vlaanderen) |
| `holiday_name` | string / NaN | Naam van de feestdag (indien van toepassing) |
| `vacation_type` | string / NaN | Type vakantie (Zomer-, Kerst-, Krokus-, etc.) |
| `calendar_day_class` | categorical | Gecombineerde dagklasse (zie hieronder) |

### Gecombineerde dagklasse (`calendar_day_class`)

Om het modeltraining te vereenvoudigen voor boomgebaseerde modellen (XGBoost, 
LightGBM) definiëren we een hiërarchische dagklasse:

1. national_holiday → officiële feestdag

2. school_vacation → schoolvakantiedag (geen feestdag)

3. other_holiday → overige herdenkingsdag (geen vak./feestdag)

4. regular_day → gewone werkdag/weekend



Dit volgt de aanpak van Tanui et al. (2025), die categorische kalenderfeatures 
gebruiken naast binaire indicatoren voor verbeterde modelnauwkeurigheid.


In [9]:
df_cal = build_calendar_master(
    df_holidays=df_holidays,
    df_vac_daily=df_vac_daily,
    start_date=DEFAULT_PROJECT_START,
    end_date=DEFAULT_PROJECT_END,
)

print(df_cal.head(10))


        date national_holiday_name other_holiday_name vacation_name  \
0 2019-01-01                   NaN                NaN           NaN   
1 2019-01-02                   NaN                NaN           NaN   
2 2019-01-03                   NaN                NaN           NaN   
3 2019-01-04                   NaN                NaN           NaN   
4 2019-01-05                   NaN                NaN           NaN   
5 2019-01-06                   NaN                NaN           NaN   
6 2019-01-07                   NaN                NaN           NaN   
7 2019-01-08                   NaN                NaN           NaN   
8 2019-01-09                   NaN                NaN           NaN   
9 2019-01-10                   NaN                NaN           NaN   

  vacation_type  is_national_holiday  is_other_holiday  is_any_holiday  \
0           NaN                    0                 0               0   
1           NaN                    0                 0               0

In [10]:
# Validatie masterkalender
print("=" * 55)
print("VALIDATIE — calendar_master")
print("=" * 55)
print(f"Totaal dagen               : {len(df_cal)}")
print(f"Datumbereik                : {df_cal['date'].min().date()} → {df_cal['date'].max().date()}")
print(f"\nDagklasse-verdeling:")
print(df_cal["calendar_day_class"].value_counts())
print(f"\nNationale feestdagen       : {df_cal['is_national_holiday'].sum()}")
print(f"Overige herdenkingsdagen   : {df_cal['is_other_holiday'].sum()}")
print(f"Schoolvakantiedagen        : {df_cal['is_school_vacation'].sum()}")
print(f"Weekenddagen               : {df_cal['is_weekend'].sum()}")

# ── Spot-check: zijn 1 januari, 11 november correct gemarkeerd? ───────────────
spot_check = df_cal[df_cal["date"].isin([
    pd.Timestamp("2022-01-01"),
    pd.Timestamp("2022-07-21"),  # Nationale feestdag
    pd.Timestamp("2022-07-15"),  # Zomervakantie
    pd.Timestamp("2022-09-05"),  # Gewone maandag
])][["date","calendar_day_class","national_holiday_name","is_school_vacation"]]
print("\nSpot-check kritieke datums:")
print(spot_check.to_string(index=False))


VALIDATIE — calendar_master
Totaal dagen               : 2922
Datumbereik                : 2019-01-01 → 2026-12-31

Dagklasse-verdeling:
calendar_day_class
regular_day         1573
weekend              612
school_vacation      594
national_holiday      72
other_holiday         71
Name: count, dtype: int64

Nationale feestdagen       : 72
Overige herdenkingsdagen   : 78
Schoolvakantiedagen        : 663
Weekenddagen               : 834

Spot-check kritieke datums:
      date calendar_day_class   national_holiday_name  is_school_vacation
2022-01-01   national_holiday      Nieuwjaarsdag 2022                   1
2022-07-15    school_vacation                     NaN                   1
2022-07-21   national_holiday Nationale feestdag 2022                   1
2022-09-05        regular_day                     NaN                   0


### Verwachte resultaten spot-check

| Datum | Verwachte klasse |
|---|---|
| 2022-01-01 | `national_holiday` (Nieuwjaarsdag) |
| 2022-07-21 | `national_holiday` (Nationale feestdag) |
| 2022-07-15 | `school_vacation` (Zomervakantie) |
| 2022-09-05 | `regular_day` |

Als de spot-check slaagt, is de merge correct verlopen.


In [11]:
# opslaan
out_path = DATA_INT / "calendar_master.parquet"
df_cal.to_parquet(out_path, index=False)
print(f"✓ Opgeslagen: {out_path}  ({len(df_cal)} rijen, {len(df_cal.columns)} kolommen)")
print(f"\nKolommen: {df_cal.columns.tolist()}")


✓ Opgeslagen: /Users/emilevandevoorde/Documents/mp_mechelen_parking_v2/data_intermediate/calendar_master.parquet  (2922 rijen, 14 kolommen)

Kolommen: ['date', 'national_holiday_name', 'other_holiday_name', 'vacation_name', 'vacation_type', 'is_national_holiday', 'is_other_holiday', 'is_any_holiday', 'is_school_vacation', 'calendar_day_class', 'year', 'month', 'weekday', 'is_weekend']


## Samenvatting

| Output | Rijen | Beschrijving |
|---|---|---|
| `calendar_holidays_daily.parquet` | ~150–180 | Alle feestdagen (national + other), 2020–2025 |
| `calendar_schoolvac_daily.parquet` | ~700–800 | Schoolvakantiedagen (Vlaanderen), 2019–2025 |
| `calendar_master.parquet` | 2.922 | Volledige dagkalender 2019–2026 met alle features |

De `calendar_master.parquet` bevat 9 kolommen klaar voor directe LEFT JOIN 
op `date_only` in de parkeerdata (zie notebook `04_mad_assembly.ipynb`):
Volgende stap

➡️ 02_data_quality_check.ipynb — Laden van timeseries_shortterm.parquet
en timeseries_longterm.parquet, anomaliedetectie, sensor-stale detectie
en basiskwaliteitsrapport.

Referenties:

Fokker et al. (2021). Long-term forecasting of off-street parking occupancy for smart cities.

Niu et al. (2023). Parking occupancy prediction under COVID-19: policy-aware TCN. Transportation Research Part A.

Tanui et al. (2025). A robust multivariate logistic regression model for smart parking occupancy prediction.

Zhang et al. (2020). PewLSTM: Periodic LSTM with weather-aware gating for parking behavior prediction. IJCAI.

Zhao et al. (2024). Research on influencing factors of urban on-street parking demand based on ensemble learning and SHAP.



***

Deze notebook is volledig zelfstandig uitvoerbaar: elke cel bouwt sequentieel voort op de vorige, alle paden zijn relatief (werkt op elk systeem), en de academische motivatie is gekoppeld aan bronnen uit literatuurreview. 
